# TD1-1 : Utilisation d'un LLM embedding

Cours Architectures IA — Nicolas Debeissat

Notebook à ouvrir dans **Google Colab**. Générer une clé Hugging Face si besoin.


## Cellule 1 — Prompt du cours

`request https://apilux3.pcloud.com/getpublinkdownload?fileid=93128819642&code=XZnqkRJZ98VUd3rEuMXAFIsEpAtnNHoKYK8k&linkpassword=&forcedownload=1 in order to get download link inside json response "path" and first "host"`


In [ ]:
import requests

PCLOUD_META_URL = (
    "https://apilux3.pcloud.com/getpublinkdownload"
    "?fileid=93128819642&code=XZnqkRJZ98VUd3rEuMXAFIsEpAtnNHoKYK8k"
    "&linkpassword=&forcedownload=1"
)

meta = requests.get(PCLOUD_META_URL, timeout=60).json()
assert meta.get("result") == 0, meta

download_url = f"https://{meta['hosts'][0]}{meta['path']}"
print("Download URL:", download_url)

response = requests.get(download_url, timeout=120)
response.raise_for_status()
response.encoding = response.apparent_encoding or "utf-8"

lines = [line.strip() for line in response.text.splitlines() if line.strip()]
print(f"Loaded {len(lines)} lines")
print(lines[:5])

## Cellule 2 — Prompt du cours

`calculate embedding of each line with model from hugging face`


In [ ]:
# !pip install -q sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
line_embeddings = model.encode(lines, show_progress_bar=True, convert_to_numpy=True)
print(line_embeddings.shape)
print(line_embeddings[0][:8])

## Cellule 3 — Prompt du cours

`Generate tsv file for https://projector.tensorflow.org/ with line_embeddings`

Ensuite : importer `vectors.tsv` + `metadata.tsv` sur https://projector.tensorflow.org/ et passer de PCA à UMAP.


In [ ]:
from pathlib import Path

out = Path("projector_lines")
out.mkdir(exist_ok=True)

vectors_path = out / "vectors.tsv"
metadata_path = out / "metadata.tsv"

with vectors_path.open("w", encoding="utf-8") as vf, metadata_path.open("w", encoding="utf-8") as mf:
        for line, emb in zip(lines, line_embeddings):
        vf.write("\t".join(str(float(x)) for x in emb) + "\n")
        mf.write(line.replace("\t", " ").replace("\n", " ") + "\n")

print("Created:", vectors_path.resolve(), metadata_path.resolve())
print("Import into https://projector.tensorflow.org/ then switch PCA → UMAP")

## Cellule 4 — Prompt du cours

`Do a K-means clustering on line_embeddings`


In [ ]:
from sklearn.cluster import KMeans

N_CLUSTERS = 10
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(line_embeddings)
centroids = kmeans.cluster_centers_

for cluster_id in range(N_CLUSTERS):
    members = [lines[i] for i, lab in enumerate(labels) if lab == cluster_id]
    print(f"Cluster {cluster_id}: {len(members)} docs")
    for sample in members[:3]:
        print("  -", sample[:120])

## Cellule 5 — Prompt du cours

`generate files for https://projector.tensorflow.org/ with the embeddings of the clusters and the first 3 lines as labels`


In [ ]:
out_c = Path("projector_clusters")
out_c.mkdir(exist_ok=True)

c_vectors = out_c / "vectors.tsv"
c_metadata = out_c / "metadata.tsv"

with c_vectors.open("w", encoding="utf-8") as vf, c_metadata.open("w", encoding="utf-8") as mf:
    mf.write("cluster\tlabel\n")
    for cluster_id in range(N_CLUSTERS):
        emb = centroids[cluster_id]
        vf.write("\t".join(str(float(x)) for x in emb) + "\n")
        members = [lines[i] for i, lab in enumerate(labels) if lab == cluster_id]
        first3 = " | ".join(m.replace("\t", " ").replace("\n", " ")[:80] for m in members[:3])
        mf.write(f"{cluster_id}\t{first3}\n")

print("Created:", c_vectors.resolve(), c_metadata.resolve())
print("Load on https://projector.tensorflow.org/")